# Coding Assessment Template — Labs 4 & 5

Use this notebook as a timed-assessment template.

## Quick checklist
- Import libraries
- Load dataset
- Inspect data
- Check missing values
- Fill missing values
- Encode categorical columns if needed
- Select target and features
- Use correlation to choose top features if required
- Scale features for SVM/KNN
- Split data
- Train models
- Evaluate accuracy / CV
- Output tree text / image if needed
- Save and load model
- Predict new observation


In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load and inspect dataset

In [ ]:
# Change filename as needed
df = pd.read_csv('your_dataset.csv')

print(df.head())
print('\nShape:', df.shape)
print('\nColumns:')
print(df.columns)
print('\nInfo:')
print(df.info())
print('\nDescribe:')
print(df.describe())


## 3. Missing values

In [ ]:
print('Missing values by column:')
print(df.isnull().sum())

print('\nRows with missing values:')
print(df[df.isnull().any(axis=1)])


In [ ]:
# Fill missing values - edit these lines for the actual dataset
# Examples:
# df['numeric_col'] = df['numeric_col'].fillna(df['numeric_col'].median())
# df['categorical_col'] = df['categorical_col'].fillna(df['categorical_col'].mode()[0])

print('Missing values after filling:')
print(df.isnull().sum())


## 4. Encode categorical columns if needed

In [ ]:
# Example: replace with actual categorical columns if needed
# cat_cols = ['purpose']
# df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# If no encoding is needed, keep a copy
df_encoded = df.copy()

print(df_encoded.head())
print(df_encoded.info())


## 5. Select target and features

In [ ]:
# Optional: drop ID-like column if it should not be used
# df_encoded = df_encoded.drop('ID', axis=1)

# Replace 'target_column' with the real target
target_column = 'target_column'

X = df_encoded.drop(target_column, axis=1)
y = df_encoded[target_column]

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFeature columns:')
print(X.columns)


## 6. Correlation and top features (Lab 5 style)

In [ ]:
corr_matrix = df_encoded.corr(numeric_only=True)
target_corr = corr_matrix[target_column].abs().sort_values(ascending=False)

print('Correlation with target:')
print(target_corr)

# Pick top N features (edit N if needed)
top_n = 10
selected_features = target_corr.drop(target_column).head(top_n).index.tolist()

print('\nSelected features:')
print(selected_features)

X_selected = df_encoded[selected_features]
print('\nX_selected shape:', X_selected.shape)


## 7. Scaling

In [ ]:
# Use for SVM / KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

print('Scaled shape:', X_scaled.shape)


## 8. Train / test split

In [ ]:
# Scaled split for SVM / KNN
X_train_scaled, X_test_scaled, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Unscaled split for Decision Tree / Random Forest / Naive Bayes
X_train, X_test, y_train2, y_test2 = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

print('Train/test split complete.')


## 9. Lab 4a / 4b quick template (Iris)

In [ ]:
# Uncomment if the assessment is the iris task
# from sklearn.datasets import load_iris
# iris = load_iris()
# X_iris = iris.data
# y_iris = iris.target
#
# dt_iris = DecisionTreeClassifier(random_state=42)
# dt_iris.fit(X_iris, y_iris)
# print(export_text(dt_iris, feature_names=list(iris.feature_names)))
#
# observation = [[5, 4, 3, 2]]
# print('DT prediction:', dt_iris.predict(observation))
# print('DT proba:', dt_iris.predict_proba(observation))
#
# rf_iris = RandomForestClassifier(random_state=42)
# rf_iris.fit(X_iris, y_iris)
# print('RF prediction:', rf_iris.predict(observation))
# print('RF proba:', rf_iris.predict_proba(observation))


## 10. Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train2)

dt_pred = dt_model.predict(X_test)
print('Decision Tree Accuracy:', accuracy_score(y_test2, dt_pred))
print(confusion_matrix(y_test2, dt_pred))
print(classification_report(y_test2, dt_pred))


## 11. Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train2)

rf_pred = rf_model.predict(X_test)
print('Random Forest Accuracy:', accuracy_score(y_test2, rf_pred))
print(confusion_matrix(y_test2, rf_pred))
print(classification_report(y_test2, rf_pred))


## 12. SVM

In [ ]:
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)
print('SVM Accuracy:', accuracy_score(y_test, svm_pred))


## 13. KNN (find best K)

In [ ]:
k_values = range(1, 21)
k_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pred_k = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred_k)
    k_accuracies.append(acc)

for k, acc in zip(k_values, k_accuracies):
    print(f'K={k}, Accuracy={acc}')

best_k = k_values[k_accuracies.index(max(k_accuracies))]
print('Best K:', best_k)

knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)
knn_pred = knn_model.predict(X_test_scaled)
print('KNN Accuracy:', accuracy_score(y_test, knn_pred))


## 14. Naive Bayes

In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train2)

nb_pred = nb_model.predict(X_test)
print('Naive Bayes Accuracy:', accuracy_score(y_test2, nb_pred))
print('Naive Bayes prediction probabilities:')
print(nb_model.predict_proba(X_test)[:10])


## 15. K-Fold cross-validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

svm_cv = cross_val_score(svm_model, X_scaled, y, cv=kf, scoring='accuracy').mean()
knn_cv = cross_val_score(knn_model, X_scaled, y, cv=kf, scoring='accuracy').mean()
dt_cv = cross_val_score(dt_model, X_selected, y, cv=kf, scoring='accuracy').mean()
nb_cv = cross_val_score(nb_model, X_selected, y, cv=kf, scoring='accuracy').mean()

print('Mean Cross-Validation Accuracies:')
print('SVM:', svm_cv)
print('KNN:', knn_cv)
print('Decision Tree:', dt_cv)
print('Naive Bayes:', nb_cv)

results = {
    'SVM': svm_cv,
    'KNN': knn_cv,
    'Decision Tree': dt_cv,
    'Naive Bayes': nb_cv
}

best_model_name = max(results, key=results.get)
print('Best Model:', best_model_name)


## 16. Decision Tree text and image

In [ ]:
tree_text = export_text(dt_model, feature_names=list(X_selected.columns))
print(tree_text)

plt.figure(figsize=(20, 10))
plot_tree(dt_model, feature_names=X_selected.columns, class_names=['0', '1'], filled=True)
plt.savefig('decision_tree.png')
plt.show()


## 17. Train final model on all data

In [ ]:
if best_model_name == 'SVM':
    final_model = SVC(probability=True, random_state=42)
    final_model.fit(X_scaled, y)
elif best_model_name == 'KNN':
    final_model = KNeighborsClassifier(n_neighbors=best_k)
    final_model.fit(X_scaled, y)
elif best_model_name == 'Decision Tree':
    final_model = DecisionTreeClassifier(random_state=42)
    final_model.fit(X_selected, y)
else:
    final_model = GaussianNB()
    final_model.fit(X_selected, y)

print('Final model trained:', best_model_name)


## 18. Save and load model

In [ ]:
joblib.dump(final_model, 'best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Model and scaler saved successfully.')

loaded_model = joblib.load('best_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
print('Model loaded successfully.')


## 19. Predict a new observation

In [ ]:
# Safer than input() in a notebook: replace the zeros with real values
new_data_df = pd.DataFrame([[0] * len(selected_features)], columns=selected_features)
print(new_data_df)

if best_model_name in ['SVM', 'KNN']:
    new_data_scaled = loaded_scaler.transform(new_data_df)
    prediction = loaded_model.predict(new_data_scaled)
    probability = loaded_model.predict_proba(new_data_scaled)
else:
    prediction = loaded_model.predict(new_data_df)
    probability = loaded_model.predict_proba(new_data_df)

print('Prediction:', prediction[0])
print('Prediction Probability:', probability)


## 20. Fast troubleshooting

- `print(df.columns)` if a column name fails
- `print(df.isnull().sum())` if the model errors on missing data
- `print(selected_features)` if your new prediction row has the wrong number of values
- Use `X_scaled` for SVM/KNN
- Use `X_selected` for Decision Tree / Random Forest / Naive Bayes
